In [1]:
import os
import pandas as pd 
import fitz
import numpy as np
import glob
from pydantic import BaseModel
from typing import List
import json
import ollama

In [2]:
pdf_location = os.path.join(os.getcwd(), "starting_point")
pdf_files = glob.glob(os.path.join(pdf_location, "*.pdf"))
print(f"Found {len(pdf_files)} PDF files.")

Found 1 PDF files.


In [3]:
import easyocr 
import re
import unicodedata

In [4]:
import layoutparser as lp
from pdf2image import convert_from_path
import detectron2
import os
from bs4 import BeautifulSoup
import json
from tqdm import tqdm

In [5]:
from grobid_client.grobid_client import GrobidClient


In [6]:
def run_grobid_batch(pdf_dir, output_dir):
    client = GrobidClient()

    client.process(
        service = "processFulltextDocument",
        input_path = pdf_dir,
        output = output_dir,
        consolidate_header = False
    )

In [7]:
xml_output_folder = "./grobid_output"
os.makedirs(xml_output_folder, exist_ok=True)


run_grobid_batch(pdf_location, xml_output_folder)

Found 1 local file(s) to process
Processing completed: 1 out of 1 files processed
Errors: 0 out of 1 files processed
⏱️  Total runtime: 2.85 seconds
🚀 Speed: 0.35 documents/second
 Throughput: 2.85 seconds/document


In [8]:
def parse_tei_to_dict(tei_file_path):
    with open(tei_file_path, 'r', encoding='utf-8') as f:
        soup = BeautifulSoup(f,'xml')

    list_bibl = soup.find('listBibl')

    if not list_bibl:
        return []

    refs = []

    for i, bibl in enumerate(list_bibl.find_all('biblStruct')):
        refs.append({
            "index": i,
            "xml_id" : bibl.get('xml:id'),
            "raw" : bibl.get_text(" ", strip=True),
        })

    return refs

In [ ]:

def parse_tei_to_dict(tei_file_path: str):
    """Parses GROBID's XML output to extract Title, Authors, and Emails."""
    with open(tei_file_path, 'r', encoding='utf-8') as f:
        soup = BeautifulSoup(f, 'xml')

    # 1. Extract Title
    title_node = soup.find('titleStmt')
    title = title_node.title.text if title_node and title_node.title else None

    # 2. Extract Authors and Emails
    authors = []
    emails = []
    
    author_nodes = soup.find_all('author')
    for author in author_nodes:
        # Reconstruct the name
        pers_name = author.find('persName')
        if pers_name:
            forename = pers_name.find('forename')
            surname = pers_name.find('surname')
            
            first = forename.text if forename else ""
            last = surname.text if surname else ""
            if first or last:
                authors.append(f"{first} {last}".strip())
        
        # Grab email if present
        email_node = author.find('email')
        if email_node:
            emails.append(email_node.text)

    return {
        "source_file": os.path.basename(tei_file_path).replace('.tei.xml', '.pdf'),
        "title": title,
        "authors": authors,
        "emails": emails
    }


In [9]:
meta = []
with open("extracted_metadata.jsonl", "w") as out_file:
    for xml_file in os.listdir(xml_output_folder):
        if xml_file.endswith('.tei.xml'):
            full_path = os.path.join(xml_output_folder, xml_file)
            data = parse_tei_to_dict(full_path)
            
            out_file.write(json.dumps(data) + '\n')
            meta.append(data)

print(f"Successfully parsed {len(meta)} documents into extracted_metadata.jsonl")

Successfully parsed 1 documents into extracted_metadata.jsonl


In [10]:
meta

[[{'index': 0,
   'xml_id': 'b0',
   'raw': 'S Datta Quantum Transport: Atom to Transistor Cambridge University Press 2005'},
  {'index': 1,
   'xml_id': 'b1',
   'raw': 'Inverse problems in biomedical imaging: modeling and methods of solution M Bertero M Piana Complex Systems in Biomedicine A Quarteroni L Formaggia A Veneziani Milan; Milano Springer 2006'},
  {'index': 2,
   'xml_id': 'b2',
   'raw': 'An introduction to full waveform inversion J Virieux A Asnaashari R Brossier L Metivier A Ribodetti W Zhou Encyclopedia of Exploration Geophysics 1 2017'},
  {'index': 3,
   'xml_id': 'b3',
   'raw': 'Spectral-element and adjoint methods in seismology J Tromp D Komatitsch Q Liu Communications in Computational Physics 3 1 2008'},
  {'index': 4,
   'xml_id': 'b4',
   'raw': 'A novel approach to environment mapping using sonar sensors and inverse problems E Dias H Vieira Neto Lecture Notes in Computer Science C Dixon K Tuyls 9287 2004 Springer'},
  {'index': 5,
   'xml_id': 'b5',
   'raw': 

In [ ]:
35+23